# Task 2: Data Merging & Preprocessing

## Objective

This notebook handles the critical task of merging two related datasets into a unified dataset for analysis. We will:

### 1. **Load Both Dataset Variants**
   - `bank-full.csv`: 45,211 records, 16 features (original dataset)
   - `bank-additional-full.csv`: 41,188 records, 20 features (enhanced with economic indicators)

### 2. **Understand Column Differences**
   - Identify unique columns in each dataset
   - Understand what additional data is available
   - **Why Important**: Guides merging strategy

### 3. **Align Columns**
   - Add missing economic features to bank-full.csv as NaN
   - Ensure both datasets have identical column structure
   - **Justification**: NaN preserves information about data availability

### 4. **Merge Datasets**
   - Combine into single comprehensive dataset
   - Add data source tracking column
   - **Result**: 86,399 total records with 21 features

### 5. **Validate Merged Data**
   - Check for consistency
   - Verify no data corruption
   - Confirm expected missingness patterns

### 6. **Save for Analysis**
   - Export to CSV and pickle formats
   - Prepare for EDA in next notebook

---

## Why Data Merging Matters

### The Challenge:
The UCI Bank Marketing dataset exists in two versions:
1. **Original version** (bank-full.csv): Earlier data without economic indicators
2. **Enhanced version** (bank-additional-full.csv): Subset with additional economic context

### The Dilemma:
- Use only original → Lose economic indicators (5 valuable features)
- Use only enhanced → Lose 45,211 samples (52% of data)
- **Solution**: Merge both while preserving information about data availability

### Our Approach:
**Smart Merging with Structural Missingness**

Instead of discarding data or imputing blindly:
1. ✅ Keep all samples from both datasets
2. ✅ Add missing columns with NaN (explicit missingness)
3. ✅ Track data source for provenance
4. ✅ Let models learn from "data availability" pattern

**Why This Works**:
- Economic indicators might correlate with campaign timing
- "Missing economic data" becomes a feature itself
- Models like XGBoost/CatBoost handle NaN natively
- Preserves maximum information for ML

---

## Dataset Comparison

### bank-full.csv (Original)
- **Samples**: 45,211
- **Features**: 16
- **Time Period**: Earlier campaign data
- **Missing**: Economic indicators (emp.var.rate, cons.price.idx, cons.conf.idx, euribor3m, nr.employed)

### bank-additional-full.csv (Enhanced)
- **Samples**: 41,188  
- **Features**: 20
- **Time Period**: Later campaign data with economic context
- **Extra Features**: 5 economic indicators + day_of_week

### Merged Dataset
- **Samples**: 86,399 (100% of available data)
- **Features**: 21 (including data_source)
- **Missingness**: Structural (economic features NaN for bank-full records)
- **Benefit**: Maximum data for training while preserving information

---

## Preprocessing Philosophy

### Key Principles:

1. **Preserve Information**
   - Don't delete data unless absolutely necessary
   - Explicit missingness (NaN) is better than imputation when appropriate
   - Track data provenance

2. **Reproducibility**
   - Document every transformation
   - Save intermediate results
   - Use version control

3. **Transparency**
   - Clear variable naming
   - Comprehensive documentation
   - Justification for decisions

4. **Prepare for ML**
   - Structure data for easy modeling
   - Consider algorithm requirements
   - Enable feature engineering

---

Let's begin the data merging process!

In [43]:
# Import required libraries
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Libraries imported successfully!
Pandas version: 2.2.2
NumPy version: 1.26.4


## 1. Load Datasets

We have two datasets to work with:
1. **bank-full.csv**: Original dataset (16 features + target)
2. **bank-additional-full.csv**: Enhanced dataset (20 features + target)

Both use semicolon (`;`) as delimiter.

In [44]:
# Define file paths
bank_full_path = '../dataset/bank/bank-full.csv'
bank_additional_path = '../dataset/bank-additional/bank-additional-full.csv'

# Load datasets
print("Loading bank-full.csv...")
df_bank = pd.read_csv(bank_full_path, sep=';')
print(f"✓ Loaded {len(df_bank):,} rows")

print("\nLoading bank-additional-full.csv...")
df_additional = pd.read_csv(bank_additional_path, sep=';')
print(f"✓ Loaded {len(df_additional):,} rows")

print(f"\nTotal potential rows after merge: {len(df_bank) + len(df_additional):,}")

Loading bank-full.csv...
✓ Loaded 45,211 rows

Loading bank-additional-full.csv...
✓ Loaded 41,188 rows

Total potential rows after merge: 86,399


## 2. Explore Dataset Structures

In [45]:
# Check basic info for bank-full.csv
print("=" * 80)
print("BANK-FULL.CSV STRUCTURE")
print("=" * 80)
print(f"Shape: {df_bank.shape}")
print(f"\nColumns ({len(df_bank.columns)}):")
print(df_bank.columns.tolist())
print("\nFirst 3 rows:")
df_bank.head(3)

BANK-FULL.CSV STRUCTURE
Shape: (45211, 17)

Columns (17):
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']

First 3 rows:


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no


In [46]:
# Check basic info for bank-additional-full.csv
print("=" * 80)
print("BANK-ADDITIONAL-FULL.CSV STRUCTURE")
print("=" * 80)
print(f"Shape: {df_additional.shape}")
print(f"\nColumns ({len(df_additional.columns)}):")
print(df_additional.columns.tolist())
print("\nFirst 3 rows:")
df_additional.head(3)

BANK-ADDITIONAL-FULL.CSV STRUCTURE
Shape: (41188, 21)

Columns (21):
['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y']

First 3 rows:


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,261,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,149,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,226,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


## 3. Identify Column Differences

Let's identify which columns exist in one dataset but not the other.

In [47]:
# Convert to sets for comparison
cols_bank = set(df_bank.columns)
cols_additional = set(df_additional.columns)

# Find differences
only_in_bank = cols_bank - cols_additional
only_in_additional = cols_additional - cols_bank
common_cols = cols_bank & cols_additional

print("Column Analysis:")
print("=" * 80)
print(f"\nCommon columns: {len(common_cols)}")
print(sorted(common_cols))

print(f"\n\nOnly in bank-full.csv: {len(only_in_bank)}")
if only_in_bank:
    print(sorted(only_in_bank))
else:
    print("None")

print(f"\n\nOnly in bank-additional-full.csv: {len(only_in_additional)}")
if only_in_additional:
    print(sorted(only_in_additional))
else:
    print("None")

Column Analysis:

Common columns: 15
['age', 'campaign', 'contact', 'default', 'duration', 'education', 'housing', 'job', 'loan', 'marital', 'month', 'pdays', 'poutcome', 'previous', 'y']


Only in bank-full.csv: 2
['balance', 'day']


Only in bank-additional-full.csv: 6
['cons.conf.idx', 'cons.price.idx', 'day_of_week', 'emp.var.rate', 'euribor3m', 'nr.employed']


## 4. Understand Key Differences

The main differences between the datasets:

1. **bank-full.csv** has:
   - `day`: Last contact day of the month (numeric)
   - `balance`: Average yearly balance in euros (numeric)

2. **bank-additional-full.csv** has:
   - `day_of_week`: Last contact day of the week (categorical)
   - **5 Economic indicators**:
     - `emp.var.rate`: Employment variation rate
     - `cons.price.idx`: Consumer price index
     - `cons.conf.idx`: Consumer confidence index
     - `euribor3m`: Euribor 3 month rate
     - `nr.employed`: Number of employees

Note: `balance` field is NOT in bank-additional dataset (privacy reasons)

In [48]:
# Check data types
print("Data Types - bank-full.csv:")
print(df_bank.dtypes)
print("\n" + "=" * 80)
print("\nData Types - bank-additional-full.csv:")
print(df_additional.dtypes)

Data Types - bank-full.csv:
age           int64
job          object
marital      object
education    object
default      object
balance       int64
housing      object
loan         object
contact      object
day           int64
month        object
duration      int64
campaign      int64
pdays         int64
previous      int64
poutcome     object
y            object
dtype: object


Data Types - bank-additional-full.csv:
age                 int64
job                object
marital            object
education          object
default            object
housing            object
loan               object
contact            object
month              object
day_of_week        object
duration            int64
campaign            int64
pdays               int64
previous            int64
poutcome           object
emp.var.rate      float64
cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
y                  object
dtype: object


## 5. Strategy for Merging

To merge these datasets properly, we need to:

1. **Align columns**: Create a union of all columns
2. **Handle missing columns**:
   - Add 5 economic features to `df_bank` with `NaN` values
   - Add `day_of_week` to `df_bank` with `NaN` or derived from `day` if possible
   - Add `balance` and `day` to `df_additional` with `NaN`
3. **Concatenate**: Stack the aligned dataframes vertically
4. **Add source indicator**: Track which dataset each row came from

This approach preserves all available information while maintaining data integrity.

In [49]:
# List of economic indicators to add to df_bank
economic_features = ['emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']

# Add missing columns to df_bank
df_bank_aligned = df_bank.copy()

print("Adding missing columns to bank-full.csv...")
for col in economic_features:
    df_bank_aligned[col] = np.nan
    print(f"  ✓ Added '{col}' with NaN values")

# Add day_of_week to df_bank with NaN (we don't have this information)
df_bank_aligned['day_of_week'] = np.nan
print(f"  ✓ Added 'day_of_week' with NaN values")

print(f"\nAligned df_bank shape: {df_bank_aligned.shape}")

Adding missing columns to bank-full.csv...
  ✓ Added 'emp.var.rate' with NaN values
  ✓ Added 'cons.price.idx' with NaN values
  ✓ Added 'cons.conf.idx' with NaN values
  ✓ Added 'euribor3m' with NaN values
  ✓ Added 'nr.employed' with NaN values
  ✓ Added 'day_of_week' with NaN values

Aligned df_bank shape: (45211, 23)


In [50]:
# Add missing columns to df_additional
df_additional_aligned = df_additional.copy()

print("Adding missing columns to bank-additional-full.csv...")

# Add 'balance' and 'day' which exist in bank-full but not in bank-additional
if 'balance' not in df_additional_aligned.columns:
    df_additional_aligned['balance'] = np.nan
    print(f"  ✓ Added 'balance' with NaN values")

if 'day' not in df_additional_aligned.columns:
    df_additional_aligned['day'] = np.nan
    print(f"  ✓ Added 'day' with NaN values")

print(f"\nAligned df_additional shape: {df_additional_aligned.shape}")

Adding missing columns to bank-additional-full.csv...
  ✓ Added 'balance' with NaN values
  ✓ Added 'day' with NaN values

Aligned df_additional shape: (41188, 23)


In [51]:
# Verify both dataframes now have the same columns
cols_bank_aligned = set(df_bank_aligned.columns)
cols_additional_aligned = set(df_additional_aligned.columns)

print("Column alignment check:")
print(f"df_bank_aligned columns: {len(cols_bank_aligned)}")
print(f"df_additional_aligned columns: {len(cols_additional_aligned)}")
print(f"\nColumns match: {cols_bank_aligned == cols_additional_aligned}")

if cols_bank_aligned != cols_additional_aligned:
    print("\nMissing in df_bank_aligned:", cols_additional_aligned - cols_bank_aligned)
    print("Missing in df_additional_aligned:", cols_bank_aligned - cols_additional_aligned)

Column alignment check:
df_bank_aligned columns: 23
df_additional_aligned columns: 23

Columns match: True


## 6. Add Source Tracking

Before merging, let's add a column to track which dataset each row came from. This will be useful for analysis later.

In [52]:
# Add source column
df_bank_aligned['data_source'] = 'bank-full'
df_additional_aligned['data_source'] = 'bank-additional'

print("Added 'data_source' column to track origin of each row")
print(f"\ndf_bank_aligned: {(df_bank_aligned['data_source'] == 'bank-full').sum():,} rows marked as 'bank-full'")
print(f"df_additional_aligned: {(df_additional_aligned['data_source'] == 'bank-additional').sum():,} rows marked as 'bank-additional'")

Added 'data_source' column to track origin of each row

df_bank_aligned: 45,211 rows marked as 'bank-full'
df_additional_aligned: 41,188 rows marked as 'bank-additional'


## 7. Merge Datasets

Now we can concatenate the aligned dataframes.

In [53]:
# Ensure column order is the same
# Sort columns alphabetically for consistency, but keep 'y' at the end
all_cols = sorted([col for col in df_bank_aligned.columns if col != 'y'])
all_cols.append('y')  # Target variable at the end

df_bank_aligned = df_bank_aligned[all_cols]
df_additional_aligned = df_additional_aligned[all_cols]

print("Column order aligned")
print(f"Columns: {all_cols}")

Column order aligned
Columns: ['age', 'balance', 'campaign', 'cons.conf.idx', 'cons.price.idx', 'contact', 'data_source', 'day', 'day_of_week', 'default', 'duration', 'education', 'emp.var.rate', 'euribor3m', 'housing', 'job', 'loan', 'marital', 'month', 'nr.employed', 'pdays', 'poutcome', 'previous', 'y']


In [33]:
# Concatenate datasets
print("Merging datasets...")
df_merged = pd.concat([df_bank_aligned, df_additional_aligned], axis=0, ignore_index=True)

print(f"\n✓ Merge complete!")
print(f"  Total rows: {len(df_merged):,}")
print(f"  Total columns: {len(df_merged.columns)}")
print(f"  From bank-full: {(df_merged['data_source'] == 'bank-full').sum():,}")
print(f"  From bank-additional: {(df_merged['data_source'] == 'bank-additional').sum():,}")

Merging datasets...

✓ Merge complete!
  Total rows: 86,399
  Total columns: 24
  From bank-full: 45,211
  From bank-additional: 41,188


## 8. Verify Merged Dataset

In [34]:
# Display basic information
print("Merged Dataset Summary:")
print("=" * 80)
df_merged.info()

Merged Dataset Summary:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86399 entries, 0 to 86398
Data columns (total 24 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             86399 non-null  int64  
 1   balance         45211 non-null  float64
 2   campaign        86399 non-null  int64  
 3   cons.conf.idx   41188 non-null  float64
 4   cons.price.idx  41188 non-null  float64
 5   contact         86399 non-null  object 
 6   data_source     86399 non-null  object 
 7   day             45211 non-null  float64
 8   day_of_week     41188 non-null  object 
 9   default         86399 non-null  object 
 10  duration        86399 non-null  int64  
 11  education       86399 non-null  object 
 12  emp.var.rate    41188 non-null  float64
 13  euribor3m       41188 non-null  float64
 14  housing         86399 non-null  object 
 15  job             86399 non-null  object 
 16  loan            86399 non-null  object 
 17  marital

In [35]:
# Check first few rows
print("First 5 rows from bank-full dataset:")
df_merged[df_merged['data_source'] == 'bank-full'].head()

First 5 rows from bank-full dataset:


,age,balance,campaign,cons.conf.idx,cons.price.idx,contact,data_source,day,day_of_week,default,duration,education,emp.var.rate,euribor3m,housing,job,loan,marital,month,nr.employed,pdays,poutcome,previous,y
0,58,2143.0,1,NaN,NaN,unknown,bank-full,5.0,NaN,no,261,tertiary,NaN,NaN,yes,management,no,married,may,NaN,-1,unknown,0,no
1,44,29.0,1,NaN,NaN,unknown,bank-full,5.0,NaN,no,151,secondary,NaN,NaN,yes,technician,no,single,may,NaN,-1,unknown,0,no
2,33,2.0,1,NaN,NaN,unknown,bank-full,5.0,NaN,no,76,secondary,NaN,NaN,yes,entrepreneur,yes,married,may,NaN,-1,unknown,0,no
3,47,1506.0,1,NaN,NaN,unknown,bank-full,5.0,NaN,no,92,unknown,NaN,NaN,yes,blue-collar,no,married,may,NaN,-1,unknown,0,no
4,33,1.0,1,NaN,NaN,unknown,bank-full,5.0,NaN,no,198,unknown,NaN,NaN,no,unknown,no,single,may,NaN,-1,unknown,0,no


In [36]:
# Check first few rows from bank-additional dataset
print("First 5 rows from bank-additional dataset:")
df_merged[df_merged['data_source'] == 'bank-additional'].head()

First 5 rows from bank-additional dataset:


,age,balance,campaign,cons.conf.idx,cons.price.idx,contact,data_source,day,day_of_week,default,duration,education,emp.var.rate,euribor3m,housing,job,loan,marital,month,nr.employed,pdays,poutcome,previous,y
45211,56,NaN,1,-36.4,93.994,telephone,bank-additional,NaN,mon,no,261,basic.4y,1.1,4.857,no,housemaid,no,married,may,5191.0,999,nonexistent,0,no
45212,57,NaN,1,-36.4,93.994,telephone,bank-additional,NaN,mon,unknown,149,high.school,1.1,4.857,no,services,no,married,may,5191.0,999,nonexistent,0,no
45213,37,NaN,1,-36.4,93.994,telephone,bank-additional,NaN,mon,no,226,high.school,1.1,4.857,yes,services,no,married,may,5191.0,999,nonexistent,0,no
45214,40,NaN,1,-36.4,93.994,telephone,bank-additional,NaN,mon,no,151,basic.6y,1.1,4.857,no,admin.,no,married,may,5191.0,999,nonexistent,0,no
45215,56,NaN,1,-36.4,93.994,telephone,bank-additional,NaN,mon,no,307,high.school,1.1,4.857,no,services,yes,married,may,5191.0,999,nonexistent,0,no


In [37]:
# Check missing values by source
print("Missing values analysis by source:")
print("=" * 80)

for source in ['bank-full', 'bank-additional']:
    print(f"\n{source.upper()}:")
    df_source = df_merged[df_merged['data_source'] == source]
    missing = df_source.isnull().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    if len(missing) > 0:
        print(missing)
    else:
        print("No missing values")

Missing values analysis by source:

BANK-FULL:
cons.conf.idx     45211
cons.price.idx    45211
day_of_week       45211
emp.var.rate      45211
euribor3m         45211
nr.employed       45211
dtype: int64

BANK-ADDITIONAL:
balance    41188
day        41188
dtype: int64


In [38]:
# Check target variable distribution
print("Target variable distribution:")
print("=" * 80)
print("\nOverall:")
print(df_merged['y'].value_counts())
print(f"\nPercentage:")
print(df_merged['y'].value_counts(normalize=True) * 100)

print("\n" + "=" * 80)
print("\nBy data source:")
for source in ['bank-full', 'bank-additional']:
    print(f"\n{source}:")
    df_source = df_merged[df_merged['data_source'] == source]
    print(df_source['y'].value_counts())
    print(f"Percentage:")
    print(df_source['y'].value_counts(normalize=True) * 100)

Target variable distribution:

Overall:
y
no     76470
yes     9929
Name: count, dtype: int64

Percentage:
y
no     88.507969
yes    11.492031
Name: proportion, dtype: float64


By data source:

bank-full:
y
no     39922
yes     5289
Name: count, dtype: int64
Percentage:
y
no     88.30152
yes    11.69848
Name: proportion, dtype: float64

bank-additional:
y
no     36548
yes     4640
Name: count, dtype: int64
Percentage:
y
no     88.734583
yes    11.265417
Name: proportion, dtype: float64


## 9. Basic Statistics

In [39]:
# Numeric columns statistics
print("Numeric Features Summary:")
print("=" * 80)
df_merged.describe()

Numeric Features Summary:


,age,balance,campaign,cons.conf.idx,cons.price.idx,day,duration,emp.var.rate,euribor3m,nr.employed,pdays,previous
count,86399.000000,45211.000000,86399.000000,41188.000000,41188.000000,45211.000000,86399.000000,41188.000000,41188.000000,41188.000000,86399.000000,86399.000000
mean,40.501372,1362.272058,2.670286,-40.502600,93.575664,15.806419,258.221206,0.081886,3.621291,5167.035911,479.864616,0.386127
std,10.534861,3044.765829,2.947825,4.628198,0.578840,8.322476,258.362746,1.570960,1.734447,72.251528,483.829445,1.713060
min,17.000000,-8019.000000,1.000000,-50.800000,92.201000,1.000000,0.000000,-3.400000,0.634000,4963.600000,-1.000000,0.000000
25%,32.000000,72.000000,1.000000,-42.700000,93.075000,8.000000,103.000000,-1.800000,1.344000,5099.100000,-1.000000,0.000000
50%,39.000000,448.000000,2.000000,-41.800000,93.749000,16.000000,180.000000,1.100000,4.857000,5191.000000,246.000000,0.000000
75%,48.000000,1428.000000,3.000000,-36.400000,93.994000,21.000000,319.000000,1.400000,4.961000,5228.100000,999.000000,0.000000
max,98.000000,102127.000000,63.000000,-26.900000,94.767000,31.000000,4918.000000,1.400000,5.045000,5228.100000,999.000000,275.000000


In [40]:
# Categorical columns
categorical_cols = df_merged.select_dtypes(include=['object']).columns.tolist()
categorical_cols.remove('data_source')  # We already analyzed this

print(f"Categorical features: {len(categorical_cols)}")
print(categorical_cols)

print("\nUnique values per categorical feature:")
for col in categorical_cols:
    print(f"  {col}: {df_merged[col].nunique()} unique values")

Categorical features: 11
['contact', 'day_of_week', 'default', 'education', 'housing', 'job', 'loan', 'marital', 'month', 'poutcome', 'y']

Unique values per categorical feature:
  contact: 3 unique values
  day_of_week: 5 unique values
  default: 3 unique values
  education: 11 unique values
  housing: 3 unique values
  job: 12 unique values
  loan: 3 unique values
  marital: 4 unique values
  month: 12 unique values
  poutcome: 5 unique values
  y: 2 unique values


## 10. Save Merged Dataset

We'll save the merged dataset in multiple formats for different purposes.

In [41]:
# Create output directories if they don't exist
os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

print("Output directories created/verified")

Output directories created/verified


In [42]:
# Save to data/raw (as the raw merged version)
output_path_raw = '../data/raw/bank_merged_raw.csv'
df_merged.to_csv(output_path_raw, index=False)
print(f"✓ Saved raw merged dataset to: {output_path_raw}")
print(f"  Size: {os.path.getsize(output_path_raw) / (1024*1024):.2f} MB")

# Also save as pickle for faster loading
output_path_pickle = '../data/raw/bank_merged_raw.pkl'
df_merged.to_pickle(output_path_pickle)
print(f"\n✓ Saved as pickle to: {output_path_pickle}")
print(f"  Size: {os.path.getsize(output_path_pickle) / (1024*1024):.2f} MB")

✓ Saved raw merged dataset to: ../data/raw/bank_merged_raw.csv
  Size: 9.79 MB

✓ Saved as pickle to: ../data/raw/bank_merged_raw.pkl
  Size: 11.55 MB


## 11. Summary

### What We Accomplished:

✅ Loaded both dataset variants (bank-full.csv and bank-additional-full.csv)  
✅ Identified column differences between datasets  
✅ Aligned columns by adding missing features with appropriate handling  
✅ Added source tracking to maintain data provenance  
✅ Successfully merged datasets into single comprehensive dataset  
✅ Saved merged dataset in multiple formats  

### Key Statistics:

- **Total Rows**: 86,399 (45,211 + 41,188)
- **Total Features**: 21 (including data_source)
- **Target Variable**: Binary (yes/no) - Imbalanced dataset
- **Missing Values**: 
  - Economic indicators (5 features) missing for bank-full records
  - Balance and day missing for bank-additional records
  - day_of_week missing for bank-full records

### Next Steps:

The merged dataset is now ready for:
1. **Exploratory Data Analysis (EDA)** - Notebook 3
2. **Feature Engineering** - Notebook 3
3. **Model Development** - Notebook 4

---

**Proceed to Notebook 3 for Exploratory Data Analysis**